In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")[:10]
OPENAI_API_KEY

'sk-proj-_f'

In [4]:
# 필수 import v1.0
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.output_parsers.base import BaseOutputParser
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [5]:
chat = ChatOpenAI()

In [7]:
chat.invoke("메시가 가장 최고의 선수인 이유는?")

AIMessage(content='메시가 가장 최고의 선수로 손꼽히는 이유는 그의 유연한 기술과 뛰어난 드리블 능력, 빠른 발 빠른 판단력, 골키퍼를 상대로 한 대결에서도 그렇고 고른 대결에서도 능통한 실력과 선제적인 실력을 발휘하는 경쾌한 경기를 펼칠수 있기 때문입니다. 또한 그는 이런 기량과 더불어 팀에 끊임없는 자극을 주는 리더십과 습관에 대한 놀라운 통찰력도 뛰어납니다. 그의 타이틀은 축구사에 있어서 거의 돌이킬수 없을 것으로 보입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 238, 'prompt_tokens': 24, 'total_tokens': 262, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DeblyxT6b9YV8OI4ezrFYnPRLZHgb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b12-fb57-7561-bf45-f9ba661643fc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 238, 'total_tokens': 262, 'input_token_details

## temperature(창의력)
- 0.0 ~ 0.3: 결정론 - 요약, 추출, 일관성, 정확성
- 0.5 ~ 0.7: 균형적 - 적당한 자연스러움, 유연성
- 0.8 ~ 1.0+: 무작위 - 창의적이고 예측 불가능

In [14]:
chat = ChatOpenAI(temperature=0)
result = chat.invoke("하늘은 무슨 색이니?")
result.content

'하늘은 파란색이에요.'

In [24]:
chat = ChatOpenAI(temperature=0.5)
result2 = chat.invoke("하늘은 무슨 색이니?")
result2.content

'하늘은 파란색이에요.'

In [21]:
chat = ChatOpenAI(temperature=1.0)
result3 = chat.invoke("하늘은 무슨 색이니?")
result3.content

'하늘은 보통 파란색이지만, 날씨나 시간대에 따라 흐리거나 붉은 등 다양한 색상을 띠기도 합니다.'

In [43]:
class NewLineOutputParser(BaseOutputParser):
    # 반드시 parse() 재정의
    def parse(self, text):
        lines = text.split("\n")
        return [line.lstrip("-123456789. ").strip() for line in lines]

In [44]:
newline_parser = NewLineOutputParser()

In [45]:
newline_parser.parse("""- 1. 햄버거\n- 2. 떡볶이\n- 3. 치킨""")

['햄버거', '떡볶이', '치킨']

In [46]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        리스트를 생성하는 기계입니다.
        요청한 모든 리스트에 개수는 최대 {max_length}개 까지만 목록으로 표시하세요.
        그 이상 초과되는 리스트는 답변하지 마세요.
    """),
    ("human", "{question}")
    
])

prompt = template.format_messages(
    max_length=5,
    question="AI를 잘하려면 어떤 것부터 공부해야 해?"
)

## 체인(Chain 생성)
- "|": 파이프 연산자로 체인을 만든다.

In [47]:
# List
first_chain = template | chat | NewLineOutputParser()

In [48]:
chain_result = first_chain.invoke({
    "max_length": 5,
    "question": "AI를 잘하려면 어떤 것부터 공부해야 해?"
})

In [49]:
chain_result

['프로그래밍 언어 (Python, Java, C++ 등)',
 '기초 수학 (선형대수, 통계학, 미적분학 등)',
 '머신러닝과 딥러닝 이론',
 '데이터 전처리 기술',
 '모델 학습과 평가 방법']

In [51]:
# RunableSequence
# 이전 단계의 출력이 다음 단계의 입력으로 자동 전달되는 파이프라인 객체
print(type(first_chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


## 1. 템플릿 생성

In [60]:
# 배열, 튜플 형태로 템플릿 생성
template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 수준의 여행 가이드입니다.
        사람들이 좋아하는 여행 장소를 많이 알고 있습니다.
        설명없이 지역 명소 이름만 목록으로 {max_length}개까지 답변하세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {place} 여행 장소 추천해줘!
    """)
])

In [61]:
second_chain = template | chat

In [62]:
trip_result = second_chain.invoke({
    "max_length": 3,
    "place": "부산"
})

In [63]:
trip_result.content

'1. 해운대 해수욕장\n2. 부산 국립해양박물관\n3. 태종대'

## 실습

In [65]:
# 저녁 메뉴 추천(양식, 중식, 한식 선택할 수 있도록)
# 체인을 생성 후 결과를 출력

template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 미슐랭 쉐프입니다.
        대중적인 음식 전문이며, 다양한 종류의 음식에 능통합니다.
        저녁 메뉴를 {max_length}개 추천해주세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {food} 음식으로 저녁메뉴 추천해줘!
    """)
])

In [66]:
third_chain = template | chat

In [69]:
food_result = third_chain.invoke({
    "max_length": 3,
    "food": "한식"
})

In [70]:
food_result.content

'물냉면, 갈비찜, 된장찌개'

In [ ]:
food_result = third_chain.invoke({
    "max_length": 3,
    "food": "한식"
})

## Stream
- 결과를 실시간 chunk 단위로 쪼개서 응답한다.

In [74]:
food_result = third_chain.stream({
    "max_length": 3,
    "food": "한식"
})

In [75]:
for chunk in food_result:
    print(chunk.content, end="", flush=True)

물냉면, 갈비찜, 된장찌개를 추천드립니다.

In [76]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        넌 주식의 트레이더야
        사용자가 주식과 관련된 질문을 하면 {max_length}개의 주식을 추천해줘

        반드시 아래의 JSON 형식으로 답변해줘
        {{
            "one": "tesla",
            "two": "samsung",
            ...
            "five": "apple"
        }}
    """),
    ("human", "{question}")
])

In [77]:
stock_chain = template | chat

In [80]:
stock_result = stock_chain.invoke({
    "max_length": 5,
    "question": "한국인들에게 가장 인기있는 최근 급등한 미국 주식 추천해줘!"
})

In [81]:
stock_result.content

'{\n    "one": "tesla",\n    "two": "amazon",\n    "three": "nvidia",\n    "four": "netflix",\n    "five": "zoom"\n}'

In [87]:
import json

json.loads(stock_result.content)

{'one': 'tesla',
 'two': 'amazon',
 'three': 'nvidia',
 'four': 'netflix',
 'five': 'zoom'}

In [88]:
class JsonOutputParser(BaseOutputParser):
    def parser(self, text):
        return json.loads(text)

In [95]:
chat = ChatOpenAI(temperature=0)

stock_chain = template | chat | JsonOutputParser()

TypeError: Can't instantiate abstract class JsonOutputParser with abstract method parse

In [93]:
stock_result = stock_chain.invoke({
    "max_length": 5,
    "question": "한국인들에게 가장 인기있는 최근 급등한 미국 주식 추천해줘!"
})

AIMessage(content='{\n    "one": "tesla",\n    "two": "amazon",\n    "three": "nvidia",\n    "four": "netflix",\n    "five": "zoom"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 145, 'total_tokens': 185, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DecnYV0f4GYvxuhjqxEe6BiXNbqn1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b4f-225d-7500-84bc-b00d41b7d452-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'output_tokens': 40, 'total_tokens': 185, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
stock_result